In [1]:
from pathlib import Path
import torch
import sys
import numpy as np
import collections
from model import REPO_ROOT, RTDETR_PYTORCH_ROOT, RTDETR_SRC_ROOT
import hashlib
import tensorrt as trt
import json
sys.path.insert(0, str(REPO_ROOT))
from polygraphy.backend.trt import (
    network_from_onnx_path, CreateConfig, engine_bytes_from_network, engine_from_bytes, TrtRunner
)

from harness.trt_runner import (
    build_engine, 
    plan_fingerprint, 
    TRTSession, 
    layer_precisions, 
    DEFAULT_TIMING_CACHE,
    ENGINES_DIR)

import onnx
from modelopt.onnx import autocast

/home/uwu/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: No module named 'lief'
Please install optional ``[onnx]`` dependencies.

In [2]:
h = lambda b: hashlib.sha256(b).hexdigest()[:16]

In [3]:
onnx_path = "/home/uwu/Personal_Projects/trt-quantize-partition/models/rtdetr/model.onnx"

In [4]:
eng1 = build_engine(onnx_path=onnx_path, tf32 = False)

In [5]:
eng2_tf32 = build_engine(onnx_path=onnx_path, tf32 = True)


In [6]:
eng3_tf32 = build_engine(onnx_path=onnx_path, tf32 = True)

In [7]:
precision_map = layer_precisions(eng2_tf32)

In [8]:
eng2_fingerprint = plan_fingerprint(eng2_tf32)
eng3_fingerprint = plan_fingerprint(eng3_tf32)
eng1_fingerprint = plan_fingerprint(eng1)

In [9]:
assert eng2_fingerprint == eng3_fingerprint, (
    "engine bytes are not reproducible across identical builds -- tactic selection is "
    "timing-dependent. Pin it with config.set_timing_cache(...) before comparing engines; "
    "until then the tf32 comparison below proves nothing.")
assert eng2_tf32 != eng1, "TF32 had no effect -- investigate before benchmarking"
print("builds are deterministic AND tf32 changes the engine ✓")

builds are deterministic AND tf32 changes the engine ✓


## FP16 export

In [10]:
from harness.precision import to_fp16
p = to_fp16('../models/rtdetr/model.onnx', '../models/rtdetr/model_fp16.onnx')
print('wrote', p, f'{p.stat().st_size/2**20:.1f} MiB')

wrote ../models/rtdetr/model_fp16.onnx 38.3 MiB


In [11]:
import collections
from harness.trt_runner import build_engine, layer_precisions
fp32 = build_engine("../models/rtdetr/model.onnx")
fp16 = build_engine("../models/rtdetr/model_fp16.onnx", tf32 = False, timing_cache = ENGINES_DIR / "timing_fp16.cache")
print(f"engine size  fp32={len(fp32)/2**20:.1f} MiB   fp16={len(fp16)/2**20:.1f} MiB")
hist = collections.Counter(d for dts in layer_precisions(fp16).values() for d in dts)
print("fp16 engine dtype histogram:", dict(hist))
assert "Half" in hist, "declared FP16 but builder produced no Half tensors"
assert len(fp16) < 0.75 * len(fp32), "FP16 engine did not shrink -- weights still FP32"
print("fp16 engine verified ✓")

engine size  fp32=89.0 MiB   fp16=40.6 MiB
fp16 engine dtype histogram: {'Half': 180, 'Float': 56, 'Int32': 3}
fp16 engine verified ✓


In [2]:
m = onnx.load("../models/rtdetr/model_fp16.onnx", load_external_data=True)

In [ ]:
for node in iter(m.graph.node):
    for input in node.input:
        print(input)
        break
    break

images


In [11]:
node = iter(m.graph.node)

In [18]:
node.__next__().input

['model.decoder.decoder.layers.0.self_attn.in_proj_weight']